# Pass 6B — BrEaST Zero-Shot External Validation

Frozen BUSI raw-ROI VGG16 evaluated on the independently audited BrEaST-Lesions-USG cohort.

**Locked protocol:** 252 cases (154 benign, 98 malignant); 15×15 elliptical dilation; 224×224 input; fixed 0.50 threshold; no retraining, fine-tuning, threshold tuning, or calibration fitting on BrEaST; 10,000 case-level bootstrap replicates.

In [ ]:
from pathlib import Path
import os, json, random
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
import cv2, numpy as np, pandas as pd, tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss, log_loss

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PASS6A_AUDIT_CSV=Path("results/pass6a_breast_dataset_audit/breast_dataset_level_audit.csv")
RAW_ROI_MODEL=Path("results/pass3b_vgg16/models/vgg16_raw_roi.keras")
OUT_DIR=Path("results/pass6b_breast_external_validation"); OUT_DIR.mkdir(parents=True,exist_ok=True)
THRESHOLD=0.50; N_BOOT=10000
assert PASS6A_AUDIT_CSV.exists(), PASS6A_AUDIT_CSV
assert RAW_ROI_MODEL.exists(), RAW_ROI_MODEL


In [ ]:
audit=pd.read_csv(PASS6A_AUDIT_CSV)
assert len(audit)==252 and audit.CaseID.nunique()==252
assert audit.Classification.value_counts().to_dict()=={"benign":154,"malignant":98}
for c in ["image_exists","mask_exists","dimensions_match","mask_binary_like","mask_nonempty"]:
    assert audit[c].all(), f"Pass 6A gate failed: {c}"
external=audit.sort_values("CaseID").reset_index(drop=True).copy()
external["true_label"]=external.Classification.map({"benign":0,"malignant":1}).astype(int)
external[["CaseID","Classification","true_label","image_path","tumor_mask_path"]].to_csv(OUT_DIR/"pass6b_locked_external_cohort.csv",index=False)
print(external.Classification.value_counts())


In [ ]:
KERNEL=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
def make_raw_roi(image_path,mask_path):
    image=np.asarray(Image.open(image_path).convert("RGB"))
    mask=np.asarray(Image.open(mask_path).convert("L"))
    d=cv2.dilate((mask>0).astype(np.uint8),KERNEL,iterations=1)
    ys,xs=np.where(d>0)
    if len(xs)==0: raise ValueError("Empty dilated mask")
    x0,x1=int(xs.min()),int(xs.max()); y0,y1=int(ys.min()),int(ys.max())
    image[d==0]=0
    crop=image[y0:y1+1,x0:x1+1]
    return cv2.resize(crop,(224,224),interpolation=cv2.INTER_AREA).astype(np.float32)/255.0

X=np.stack([make_raw_roi(r.image_path,r.tumor_mask_path) for r in external]).astype(np.float32)
assert X.shape==(252,224,224,3)


In [ ]:
model=tf.keras.models.load_model(RAW_ROI_MODEL,compile=False)
for layer in model.layers: layer.trainable=False
assert len(model.trainable_weights)==0
assert model.input_shape[-3:]==(224,224,3)

prob=np.asarray(model.predict(X,batch_size=32,verbose=1)).reshape(-1)
assert prob.shape==(252,) and np.isfinite(prob).all() and ((prob>=0)&(prob<=1)).all()
pred=(prob>=THRESHOLD).astype(int)

predictions=external.copy()
predictions["prob_malignant"]=prob
predictions["predicted_label"]=pred
predictions["predicted_class"]=np.where(pred==1,"malignant","benign")
predictions["correct"]=predictions.true_label==pred
predictions.to_csv(OUT_DIR/"pass6b_predictions.csv",index=False)


In [ ]:
y=external.true_label.to_numpy(); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
acc=accuracy_score(y,pred); bal=balanced_accuracy_score(y,pred)
prec=precision_score(y,pred,zero_division=0); sens=recall_score(y,pred,zero_division=0)
spec=tn/(tn+fp) if tn+fp else np.nan
f1=2*prec*sens/(prec+sens) if prec+sens else 0
roc=roc_auc_score(y,prob); pr=average_precision_score(y,prob)

metrics=pd.DataFrame([dict(dataset="BrEaST-Lesions-USG",n=252,n_benign=154,n_malignant=98,threshold=THRESHOLD,accuracy=acc,balanced_accuracy=bal,precision=prec,sensitivity=sens,specificity=spec,f1=f1,roc_auc=roc,pr_auc=pr,tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))])
metrics.to_csv(OUT_DIR/"pass6b_external_metrics.csv",index=False)
pd.DataFrame([[tn,fp],[fn,tp]],index=["true_benign","true_malignant"],columns=["pred_benign","pred_malignant"]).to_csv(OUT_DIR/"pass6b_confusion_matrix.csv")
print(metrics.T)


In [ ]:
def metric_dict(y,p,q):
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    a=(y==p).mean(); se=tp/(tp+fn) if tp+fn else np.nan; sp=tn/(tn+fp) if tn+fp else np.nan
    prc=tp/(tp+fp) if tp+fp else np.nan
    f=2*prc*se/(prc+se) if np.isfinite(prc) and np.isfinite(se) and prc+se else np.nan
    try: ra=roc_auc_score(y,q)
    except ValueError: ra=np.nan
    try: pa=average_precision_score(y,q)
    except ValueError: pa=np.nan
    return {"accuracy":a,"balanced_accuracy":(se+sp)/2,"precision":prc,"sensitivity":se,"specificity":sp,"f1":f,"roc_auc":ra,"pr_auc":pa}

point=metric_dict(y,pred,prob)
rng=np.random.default_rng(SEED); boot={k:[] for k in point}
for _ in range(N_BOOT):
    i=rng.integers(0,len(y),len(y)); m=metric_dict(y[i],pred[i],prob[i])
    for k in boot: boot[k].append(m[k])
rows=[]
for k,v in boot.items():
    a=np.asarray(v,dtype=float); a=a[np.isfinite(a)]
    rows.append({"metric":k,"estimate":point[k],"ci95_low":np.quantile(a,.025),"ci95_high":np.quantile(a,.975),"n_bootstrap_finite":len(a)})
pd.DataFrame(rows).to_csv(OUT_DIR/"pass6b_bootstrap_ci.csv",index=False)
print(pd.DataFrame(rows))


In [ ]:
# Descriptive calibration diagnostics; no calibration is fitted.
cal=pd.DataFrame([{"n":252,"brier_score":brier_score_loss(y,prob),"log_loss":log_loss(y,prob,labels=[0,1]),"threshold":THRESHOLD}])
cal.to_csv(OUT_DIR/"pass6b_calibration_summary.csv",index=False)

bins=np.linspace(0,1,11); labels=[f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(10)]
r=pd.DataFrame({"prob_malignant":prob,"true_label":y})
r["bin"]=pd.cut(r.prob_malignant,bins=bins,labels=labels,include_lowest=True)
rel=r.groupby("bin",observed=False).agg(n=("true_label","size"),mean_predicted_probability=("prob_malignant","mean"),observed_malignant_rate=("true_label","mean")).reset_index()
rel.to_csv(OUT_DIR/"pass6b_reliability_table.csv",index=False)
print(cal); print(rel)


In [ ]:
manifest={"analysis":"Pass 6B BrEaST zero-shot external validation","n_cases":252,"n_benign":154,"n_malignant":98,"model":str(RAW_ROI_MODEL),"model_condition":"BUSI raw ROI VGG16","external_adaptation":False,"retraining":False,"fine_tuning":False,"threshold":THRESHOLD,"threshold_tuned_on_external_data":False,"roi_kernel":"15x15 elliptical","input_size":"224x224x3","bootstrap_unit":"BrEaST CaseID","bootstrap_replicates":N_BOOT,"seed":SEED}
with open(OUT_DIR/"pass6b_manifest.json","w") as f: json.dump(manifest,f,indent=2)
print("PASS 6B COMPLETE")
